In [1]:
# ============================================================================
# CELL 1: Preprocess and Save All Compounds (Run Once)
# ============================================================================

import os
import pandas as pd
import torch
import numpy as np
from rdkit import Chem
from scipy import sparse as sp
from torch_geometric.data import Data
from tqdm import tqdm  # for progress bar

class SMILESToPyG:
    """
    Converts SMILES strings to PyG Data objects.
    Replicates the DGL implementation exactly.
    """
    
    def __init__(self, pos_enc_dim=8, use_chirality=True, explicit_H=False):
        self.pos_enc_dim = pos_enc_dim
        self.use_chirality = use_chirality
        self.explicit_H = explicit_H
    
    def one_of_k_encoding(self, x, allowable_set):
        """Strict encoding - raises exception if x not in allowable_set"""
        if x not in allowable_set:
            raise Exception("input {0} not in allowable set{1}:".format(
                x, allowable_set))
        return [x == s for s in allowable_set]
    
    def one_of_k_encoding_unk(self, x, allowable_set):
        """Maps inputs not in the allowable set to the last element."""
        if x not in allowable_set:
            x = allowable_set[-1]
        return [x == s for s in allowable_set]
    
    def atom_features(self, atom):
        """Generate atom features including atom symbol(17),degree(7),formal charge(1),
        radical electrons(1),hybridization(6),aromatic(1),hydrogen atoms attached(5),Chirality(3)
        """
        symbol = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br', 'I', 'B', 'Si', 'Fe', 'Zn', 'Cu', 'Mn', 'Mo', 'other']  # 17-dim
        degree = [0, 1, 2, 3, 4, 5, 6]  # 7-dim
        hybridizationType = [Chem.rdchem.HybridizationType.SP,
                             Chem.rdchem.HybridizationType.SP2,
                             Chem.rdchem.HybridizationType.SP3,
                             Chem.rdchem.HybridizationType.SP3D,
                             Chem.rdchem.HybridizationType.SP3D2,
                             'other']  # 6-dim
        
        results = self.one_of_k_encoding_unk(atom.GetSymbol(), symbol) + \
                  self.one_of_k_encoding(atom.GetDegree(), degree) + \
                  [atom.GetFormalCharge(), atom.GetNumRadicalElectrons()] + \
                  self.one_of_k_encoding_unk(atom.GetHybridization(), hybridizationType) + [
                      atom.GetIsAromatic()]  # 17+7+2+6+1=33
        
        if not self.explicit_H:
            results = results + self.one_of_k_encoding_unk(atom.GetTotalNumHs(),
                                                          [0, 1, 2, 3, 4])  # 33+5=38
        if self.use_chirality:
            try:
                results = results + self.one_of_k_encoding_unk(
                    atom.GetProp('_CIPCode'),
                    ['R', 'S']) + [atom.HasProp('_ChiralityPossible')]
            except:
                results = results + [False, False] + [atom.HasProp('_ChiralityPossible')]  # 38+3 =41
        
        return results
    
    def bond_features(self, bond):
        """Generate bond features including bond type(4), conjugated(1), in ring(1), stereo(4)"""
        bt = bond.GetBondType()
        bond_feats = [
            bt == Chem.rdchem.BondType.SINGLE, bt == Chem.rdchem.BondType.DOUBLE,
            bt == Chem.rdchem.BondType.TRIPLE, bt == Chem.rdchem.BondType.AROMATIC,
            bond.GetIsConjugated(),
            bond.IsInRing()
        ]
        if self.use_chirality:
            bond_feats = bond_feats + self.one_of_k_encoding_unk(
                str(bond.GetStereo()),
                ["STEREONONE", "STEREOANY", "STEREOZ", "STEREOE"])
        return np.array(bond_feats).astype(int)
    
    def laplacian_positional_encoding(self, edge_index, num_nodes):
        """
        Graph positional encoding v/ Laplacian eigenvectors
        Exactly replicates the DGL author's implementation
        """
        # Laplacian
        edge_index_np = edge_index.numpy()
        A = sp.csr_matrix(
            (np.ones(edge_index_np.shape[1]),
             (edge_index_np[0], edge_index_np[1])),
            shape=(num_nodes, num_nodes)
        ).astype(float)
        
        # CRITICAL: Use column sum (axis=0) to match author's g.in_degrees()
        degrees = np.array(A.sum(axis=0)).flatten()
        
        # Create D^(-1/2) diagonal matrix with clipping to avoid division by zero
        N = sp.diags(degrees.clip(1) ** -0.5, dtype=float)
        
        # Normalized Laplacian: L = I - D^(-1/2) * A * D^(-1/2)
        L = sp.eye(num_nodes) - N * A * N
        
        # Eigenvectors with numpy
        EigVal, EigVec = np.linalg.eig(L.toarray())
        idx = EigVal.argsort()  # increasing order
        EigVal, EigVec = EigVal[idx], np.real(EigVec[:, idx])
        
        # Pad if needed
        if EigVec.shape[1] < self.pos_enc_dim + 1:
            PadVec = np.zeros((EigVec.shape[0], self.pos_enc_dim + 1 - EigVec.shape[1]), dtype=EigVec.dtype)
            EigVec = np.concatenate((EigVec, PadVec), 1)
        
        # Return positional encoding (skip first eigenvector which is constant)
        lap_pos_enc = torch.from_numpy(EigVec[:, 1:self.pos_enc_dim + 1]).float()
        
        return lap_pos_enc
    
    def convert(self, smiles):
        """
        Convert SMILES string to PyG Data object
        Replicates smiles_to_graph() from DGL implementation
        """
        try:
            mol = Chem.MolFromSmiles(smiles)
        except:
            raise RuntimeError("SMILES cannot been parsed!")
        
        # Add nodes
        num_atoms = mol.GetNumAtoms()
        
        # Get atom features
        atom_feats = np.array([self.atom_features(a) for a in mol.GetAtoms()])
        
        if self.use_chirality:
            chiralcenters = Chem.FindMolChiralCenters(mol, force=True, includeUnassigned=True,
                                                      useLegacyImplementation=False)
            chiral_arr = np.zeros([num_atoms, 3])
            for (i, rs) in chiralcenters:
                if rs == 'R':
                    chiral_arr[i, 0] = 1
                elif rs == 'S':
                    chiral_arr[i, 1] = 1
                else:
                    chiral_arr[i, 2] = 1
            atom_feats = np.concatenate([atom_feats, chiral_arr], axis=1)
        
        x = torch.tensor(atom_feats, dtype=torch.float32)
        
        # Add edges
        src_list = []
        dst_list = []
        bond_feats_all = []
        num_bonds = mol.GetNumBonds()
        for i in range(num_bonds):
            bond = mol.GetBondWithIdx(i)
            u = bond.GetBeginAtomIdx()
            v = bond.GetEndAtomIdx()
            bond_feats = self.bond_features(bond)
            src_list.extend([u, v])
            dst_list.extend([v, u])
            bond_feats_all.append(bond_feats)
            bond_feats_all.append(bond_feats)
        
        edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
        edge_attr = torch.tensor(np.array(bond_feats_all), dtype=torch.float32)
        
        # Compute Laplacian positional encoding
        laplacian_pe = self.laplacian_positional_encoding(edge_index, num_atoms)
        
        # Create PyG Data object
        data = Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            laplacian_pe=laplacian_pe,
            smiles=smiles
        )
        
        return data


def preprocess_and_save_compounds():
    """
    Read Davis_compound_mapping.csv and save all compounds as PyG graphs
    """
    # Paths
    compound_csv = 'data/Davis/Davis_compound_mapping.csv'
    output_dir = 'drugs'
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Read compound mapping
    print(f"Reading {compound_csv}...")
    df = pd.read_csv(compound_csv)
    print(f"Found {len(df)} unique compounds")
    
    # Initialize converter
    converter = SMILESToPyG(pos_enc_dim=8, use_chirality=True, explicit_H=False)
    
    # Process each compound
    success_count = 0
    failed_compounds = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing compounds"):
        compound_id = row['COMPOUND_ID']
        smiles = row['COMPOUND_SMILES']
        
        try:
            # Convert SMILES to PyG graph
            compound_graph = converter.convert(smiles)
            
            # Save as .pt file
            save_path = os.path.join(output_dir, f'{compound_id}.pt')
            torch.save(compound_graph, save_path)
            
            success_count += 1
            
        except Exception as e:
            print(f"\nError processing compound {compound_id}: {e}")
            failed_compounds.append(compound_id)
            continue
    
    print(f"\n{'='*60}")
    print(f"Processing Complete!")
    print(f"Successfully processed: {success_count}/{len(df)} compounds")
    print(f"Failed: {len(failed_compounds)} compounds")
    if failed_compounds:
        print(f"Failed compound IDs: {failed_compounds}")
    print(f"Saved to: {output_dir}/")
    print(f"{'='*60}")

# RUN THIS TO PROCESS ALL COMPOUNDS
preprocess_and_save_compounds()

Reading data/Davis/Davis_compound_mapping.csv...
Found 68 unique compounds


Processing compounds: 100%|██████████| 68/68 [00:00<00:00, 253.14it/s]


Processing Complete!
Successfully processed: 68/68 compounds
Failed: 0 compounds
Saved to: drugs/


# from this point everything is dummy

In [4]:
# ============================================================================
# CELL 2: Test Loading a Single Compound (Verify it works)
# ============================================================================

def test_load_compound(compound_id):
    """
    Test loading a single compound graph
    """
    file_path = f'drugs/{compound_id}.pt'
    
    if not os.path.exists(file_path):
        print(f"Error: {file_path} does not exist!")
        return None
    
    compound_graph = torch.load(file_path,weights_only=False)
    
    print(f"Loaded compound {compound_id}:")
    print(f"  - Number of atoms: {compound_graph.x.shape[0]}")
    print(f"  - Atom features shape: {compound_graph.x.shape}")
    print(f"  - Number of bonds: {compound_graph.edge_index.shape[1] // 2}")
    print(f"  - Edge features shape: {compound_graph.edge_attr.shape}")
    print(f"  - Laplacian PE shape: {compound_graph.laplacian_pe.shape}")
    print(f"  - SMILES: {compound_graph.smiles}")
    
    return compound_graph

# Test with first compound from CSV
df_test = pd.read_csv('data/Davis/Davis_compound_mapping.csv')
first_compound_id = df_test.iloc[0]['COMPOUND_ID']
print(f"Testing with compound ID: {first_compound_id}\n")
test_graph = test_load_compound(first_compound_id)

Testing with compound ID: 11314340

Loaded compound 11314340:
  - Number of atoms: 27
  - Atom features shape: torch.Size([27, 44])
  - Number of bonds: 30
  - Edge features shape: torch.Size([60, 10])
  - Laplacian PE shape: torch.Size([27, 8])
  - SMILES: CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC=C4)N
